# Experiment 6 — LLM-Only Data Cleaning with Dual Evaluation

**Setup:**
- 25 query rows from Dataset 1 with at least 1 missing target attribute
- KB: 100 rows (all matching ground truth rows + random fill)
- 5 target attributes: `bus_type`, `model_number`, `model`, `read_speed_mb_s`, `height_mm`

**Prediction:** Few-shot LLM extraction from product title+description (no KB retrieval)

**Evaluation:**
1. Standard string/numeric match
2. LLM-based evaluation — LLM judges using GT + KB context

## 1. Setup

In [48]:
import sys
sys.path.insert(0, '/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI/venv/lib64/python3.12/site-packages')
sys.path.insert(0, '/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI/venv/lib/python3.12/site-packages')
sys.path.append('/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI')

import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import pandas as pd
import numpy as np
import random
import time
import json
import re
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

random.seed(42)
np.random.seed(42)

TARGET_ATTRIBUTES = ["bus_type", "model_number", "model", "read_speed_mb_s", "write_speed_mb_s","height_mm", "width_mm"]
NUMERIC_ATTRIBUTES = {"read_speed_mb_s", "write_speed_mb_s", "height_mm", "width_mm"}

print("Setup complete.")

Setup complete.


## 2. Load Datasets

In [49]:
df1 = pd.read_json("normalized_products/dataset_1_normalized.json")
df2 = pd.read_json("normalized_products/dataset_2_normalized.json")
df3 = pd.read_json("normalized_products/dataset_3_normalized.json")
df4 = pd.read_json("normalized_products/dataset_4_normalized.json")
kb_full = pd.concat([df2, df3, df4], ignore_index=True)

print(f"Dataset 1: {len(df1)} rows")
print(f"KB full:   {len(kb_full)} rows")

Dataset 1: 812 rows
KB full:   2200 rows


## 3. Build Evaluation Set (25 query rows)

In [50]:
def get_ground_truth(cluster_id, attribute):
    matches = kb_full[kb_full["cluster_id"] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attribute)
        if pd.notna(val) and str(val).strip().lower() not in {"", "none", "nan"}:
            return str(val).strip()
    return None

selected_rows = []
for idx, row in df1.iterrows():
    missing_attrs = []
    for attr in TARGET_ATTRIBUTES:
        if pd.isna(row.get(attr)):
            gt = get_ground_truth(row["cluster_id"], attr)
            if gt is not None:
                missing_attrs.append((attr, gt))
    if missing_attrs:
        selected_rows.append({"df1_idx": idx, "missing_attrs": missing_attrs})
    if len(selected_rows) == 25:
        break

eval_records = []
for item in selected_rows:
    for attr, gt in item["missing_attrs"]:
        eval_records.append({
            "df1_idx": item["df1_idx"],
            "attribute": attr,
            "ground_truth": gt,
            "is_numeric": attr in NUMERIC_ATTRIBUTES
        })

eval_df = pd.DataFrame(eval_records)
query_indices = [item["df1_idx"] for item in selected_rows]
query_df = df1.loc[query_indices].copy()

print(f"Query rows:       {len(query_df)}")
print(f"Total eval tasks: {len(eval_df)}")
print()
print("Tasks per attribute:")
print(eval_df["attribute"].value_counts())

Query rows:       25
Total eval tasks: 39

Tasks per attribute:
attribute
model_number        16
bus_type            10
read_speed_mb_s      4
height_mm            3
width_mm             3
write_speed_mb_s     2
model                1
Name: count, dtype: int64


## 4. Build Knowledge Base (100 rows: GT rows + random fill)

In [51]:
query_cluster_ids = query_df["cluster_id"].tolist()
relevant_kb = kb_full[kb_full["cluster_id"].isin(query_cluster_ids)].copy()
remaining_needed = max(0, 100 - len(relevant_kb))
irrelevant_kb = kb_full[~kb_full["cluster_id"].isin(query_cluster_ids)]
random_fill = irrelevant_kb.sample(n=min(remaining_needed, len(irrelevant_kb)), random_state=42)
kb = pd.concat([relevant_kb, random_fill], ignore_index=True)

print(f"Relevant KB rows (with GT): {len(relevant_kb)}")
print(f"Random fill rows:           {len(random_fill)}")
print(f"Final KB size:              {len(kb)}")

Relevant KB rows (with GT): 69
Random fill rows:           31
Final KB size:              100


In [52]:
relevant_kb.head(5)

,id,brand,title,description,price,priceCurrency,cluster_id,url,title_description,model,...,bus_type,interface_type,width_mm,length_mm,height_mm,weight_g,storage_connection_type,memory_type,color,form_factor
0,19126355,Gigabyte,Gigabyte NVIDIA GeForce RTX 3080 10GB GAMING O...,Gigabyte NVIDIA GeForce RTX 3080 GAMING OC 10G...,99999.99,GBP,1002037,https://www.scan.co.uk/products/gigabyte-nvidi...,Gigabyte NVIDIA GeForce RTX 3080 10GB GAMING O...,GeForce RTX 3080 10GB GAMING OC,...,None,None,NaN,NaN,NaN,NaN,None,GDDR6X,None,None
2,46775597,Corsair,CORSAIR - Force Series MP510 960GB M.2 SSD PCI...,CORSAIR Force Series MP510 960GB M.2 SSD PCIe ...,NaN,None,1007272,https://www.dindator.se/corsair-force-series-m...,CORSAIR - Force Series MP510 960GB M.2 SSD PCI...,Force Series MP510,...,PCIe 3.0 x4,NVMe,NaN,NaN,NaN,NaN,M.2,None,None,M.2
3,33563069,None,"4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...","4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...",169.18,USD,1014052,https://www.avadirect.com/4TB-Exos-7E8-ST4000N...,"4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...",Exos 7E8,...,SATA III,SATA,NaN,NaN,NaN,NaN,3.5-Inch SATA,None,None,3.5-Inch
4,56179024,Asus,ASUS GeForce GTX 1650 4GB Phoenix Boost Graphi...,"Available @ CCL, this ASUS GeForce GTX 1650 Ph...",140.99,GBP,1014152,https://www.cclonline.com/product/279214/90YV0...,ASUS GeForce GTX 1650 4GB Phoenix Boost Graphi...,GeForce GTX 1650 4GB Phoenix Boost,...,None,None,NaN,NaN,NaN,NaN,None,None,None,None
5,84470461,Asus,Asus Dual GeForce GTX 1650 OC 4GB 128-Bit Edit...,896 CUDA CoresTuring Architecture4GB of GDDR5 ...,731.50,AED,1021594,https://pcdubai.com/product/asus-dual-geforce-...,Asus Dual GeForce GTX 1650 OC 4GB 128-Bit Edit...,Dual GeForce GTX 1650 OC 4GB 128-Bit Edition,...,None,None,NaN,NaN,NaN,NaN,None,GDDR5,None,None


In [47]:
relevant_kb.head(5)

,id,brand,title,description,price,priceCurrency,cluster_id,url,title_description,model,...,bus_type,interface_type,width_mm,length_mm,height_mm,weight_g,storage_connection_type,memory_type,color,form_factor
0,19126355,Gigabyte,Gigabyte NVIDIA GeForce RTX 3080 10GB GAMING O...,Gigabyte NVIDIA GeForce RTX 3080 GAMING OC 10G...,99999.99,GBP,1002037,https://www.scan.co.uk/products/gigabyte-nvidi...,Gigabyte NVIDIA GeForce RTX 3080 10GB GAMING O...,GeForce RTX 3080 10GB GAMING OC,...,None,None,NaN,NaN,NaN,NaN,None,GDDR6X,None,None
2,46775597,Corsair,CORSAIR - Force Series MP510 960GB M.2 SSD PCI...,CORSAIR Force Series MP510 960GB M.2 SSD PCIe ...,NaN,None,1007272,https://www.dindator.se/corsair-force-series-m...,CORSAIR - Force Series MP510 960GB M.2 SSD PCI...,Force Series MP510,...,PCIe 3.0 x4,NVMe,NaN,NaN,NaN,NaN,M.2,None,None,M.2
3,33563069,None,"4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...","4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...",169.18,USD,1014052,https://www.avadirect.com/4TB-Exos-7E8-ST4000N...,"4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/...",Exos 7E8,...,SATA III,SATA,NaN,NaN,NaN,NaN,3.5-Inch SATA,None,None,3.5-Inch
5,84470461,Asus,Asus Dual GeForce GTX 1650 OC 4GB 128-Bit Edit...,896 CUDA CoresTuring Architecture4GB of GDDR5 ...,731.50,AED,1021594,https://pcdubai.com/product/asus-dual-geforce-...,Asus Dual GeForce GTX 1650 OC 4GB 128-Bit Edit...,Dual GeForce GTX 1650 OC 4GB 128-Bit Edition,...,None,None,NaN,NaN,NaN,NaN,None,GDDR5,None,None
6,12318405,Kingston,Kingston Data Traveler Vault Privacy 3.016 Gb,Kingston DataTraveler Vault Privacy 3.0Kingsto...,NaN,AUD,1023407,https://zeppgear.com.au/products/kingston-data...,Kingston Data Traveler Vault Privacy 3.016 Gb....,Data Traveler Vault Privacy 3.0,...,USB 3.0,None,NaN,NaN,NaN,NaN,None,None,None,None


## 5. Sanity Check — Ground Truth in KB

In [53]:
print("=== SANITY CHECK: Ground truth present in KB ===\n")
found = 0
not_found = 0

for _, task in eval_df.iterrows():
    idx = task["df1_idx"]
    attr = task["attribute"]
    gt = task["ground_truth"]
    cluster_id = query_df.loc[idx, "cluster_id"]
    kb_matches = kb[kb["cluster_id"] == cluster_id]
    val_in_kb = any(
        pd.notna(r.get(attr)) and str(r.get(attr)).strip() == str(gt).strip()
        for _, r in kb_matches.iterrows()
    )
    status = "✓" if val_in_kb else "✗"
    found += int(val_in_kb)
    not_found += int(not val_in_kb)
    print(f"{status} Row {idx} | {attr:<20} | GT: {str(gt):<30} | KB matches: {len(kb_matches)}")

print(f"\nSummary: {found} found, {not_found} not found")
print(f"Coverage: {100*found/(found+not_found):.1f}%")

=== SANITY CHECK: Ground truth present in KB ===

✓ Row 0 | model_number         | GT: GV-N3080GAMING OC-10GD         | KB matches: 3
✓ Row 2 | model_number         | GT: CSSD-F960GBMP510               | KB matches: 3
✓ Row 2 | read_speed_mb_s      | GT: 3480.0                         | KB matches: 3
✓ Row 2 | write_speed_mb_s     | GT: 3000.0                         | KB matches: 3
✓ Row 3 | read_speed_mb_s      | GT: 226.0                          | KB matches: 3
✓ Row 4 | width_mm             | GT: 433.0                          | KB matches: 3
✓ Row 5 | bus_type             | GT: PCIe 3.0 x16                   | KB matches: 3
✓ Row 5 | model_number         | GT: 90YV0CV2-M0NA00                | KB matches: 3
✓ Row 6 | bus_type             | GT: USB 3.0                        | KB matches: 2
✓ Row 8 | model_number         | GT: GV-N166SOC-6GD                 | KB matches: 3
✓ Row 9 | bus_type             | GT: PCIe 3.0 x16                   | KB matches: 3
✓ Row 9 | model_number    

## 6. LLM Setup

In [54]:
predict_model = ChatOllama(model="llama3.1:8b", temperature=0)
eval_model = ChatOllama(model="llama3.1:8b", temperature=0)

test = predict_model.invoke("Say OK")
print("Ollama OK:", repr(test.content[:20]))

Ollama OK: 'OK'


## 7. Few-Shot LLM Prediction

In [39]:
def build_few_shot_prompt(product_text):
    return f"""You are a product data expert specializing in computer hardware.
Extract specific attributes from product text. Use ONLY information explicitly stated in the text.
The model_number is a manufacturer SKU — alphanumeric code, often in parentheses or at end of title.
Return ONLY a valid JSON object, nothing else.

--- EXAMPLE 1 ---
Product: WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM SATA 6Gb/s 256MB Cache 3.5 Inch - WD60EZAZ. Description: WD Blue 6TB Desktop Hard Disk Drive - 5400 RPM SATA 6Gb/s 256MB Cache 3.5 Inch - WD60EZAZ
Output: {{"bus_type": "SATA III", "model_number": "WD60EZAZ", "model": "WD Blue", "read_speed_mb_s": null, "height_mm": null}}

--- EXAMPLE 2 ---
Product: CORSAIR Force Series MP510 960GB M.2 SSD PCIe Gen3 x4, NVMe, up to 3480/3000MB/s (CSSD-F960GBMP510). Description: PCI Express 3.0 x4, 960GB
Output: {{"bus_type": "PCIe 3.0 x4", "model_number": "CSSD-F960GBMP510", "model": "Force Series MP510", "read_speed_mb_s": 3480.0, "height_mm": null}}

--- EXAMPLE 3 ---
Product: 4TB Exos 7E8 ST4000NM0115, 7200 RPM, SATA 6Gb/s, 512e, 128MB cache, 3.5-Inch HDD
Output: {{"bus_type": "SATA III", "model_number": "ST4000NM0115", "model": "Exos 7E8", "read_speed_mb_s": null, "height_mm": null}}

--- EXAMPLE 4 ---
Product: XPG GAMMIX 1TB S11 Pro PCIe NVMe Gen3x4 M.2 2280 SSD (AGAMMIXS11P-1TT-C). Description: PCIe Gen3x4, NVMe 1.3, read/write speeds up to 3500/3000MB per second.
Output: {{"bus_type": "PCIe 3.0 x4", "model_number": "AGAMMIXS11P-1TT-C", "model": "GAMMIX S11 Pro", "read_speed_mb_s": 3500.0, "height_mm": null}}

--- EXAMPLE 5 ---
Product: Gigabyte AMD Radeon RX 5600 XT GAMING OC Video Card GV-R56XTGAMING OC-6GD. Description: Interface: PCI-E 4.0 x 16, 6GB GDDR6
Output: {{"bus_type": "PCIe 4.0 x16", "model_number": "GV-R56XTGAMING OC-6GD", "model": "Radeon RX 5600 XT GAMING OC", "read_speed_mb_s": null, "height_mm": null}}

--- NOW EXTRACT ---
Product: {str(product_text)[:600]}
Output:"""


def extract_with_few_shot(df, predict_model):
    results = []
    total = len(df)
    t0 = time.time()
    for i, (idx, row) in enumerate(df.iterrows()):
        text = row.get("title_description", "")
        if not text or pd.isna(text):
            results.append({"df1_idx": idx, "bus_type": None,
                            "model_number": None, "model": None,
                            "read_speed_mb_s": None, "height_mm": None})
            continue
        prompt = build_few_shot_prompt(str(text))
        response = predict_model.invoke([HumanMessage(content=prompt)])
        response_text = response.content.strip()
        try:
            match = re.search(r'\{.*\}', response_text, re.DOTALL)
            data = json.loads(match.group(0)) if match else {}
        except:
            data = {}
        results.append({
            "df1_idx": idx,
            "bus_type": data.get("bus_type"),
            "model_number": data.get("model_number"),
            "model": data.get("model"),
            "read_speed_mb_s": data.get("read_speed_mb_s"),
            "height_mm": data.get("height_mm")
        })
        elapsed = time.time() - t0
        eta = (elapsed / (i+1)) * (total - i - 1)
        print(f"  [{i+1}/{total}] Row {idx} | ETA: {eta/60:.1f}min | {data}")
    return pd.DataFrame(results).set_index("df1_idx")


print("Running few-shot LLM extraction on 25 rows...")
t0 = time.time()
extracted_df = extract_with_few_shot(query_df, predict_model)
print(f"\nDone in {time.time()-t0:.1f}s")
print(extracted_df.to_string())

Running few-shot LLM extraction on 25 rows...
  [1/25] Row 0 | ETA: 7.9min | {'bus_type': None, 'model_number': 'GV-N3080GAMING OC-10GD', 'model': 'GeForce RTX 3080 Gaming OC', 'read_speed_mb_s': None, 'height_mm': None}
  [2/25] Row 2 | ETA: 5.2min | {'bus_type': 'PCIe 3.0 x4', 'model_number': None, 'model': 'Force MP510', 'read_speed_mb_s': None, 'height_mm': None}
  [3/25] Row 3 | ETA: 4.2min | {'bus_type': 'SATA III', 'model_number': 'ST4000NM0115', 'model': 'EXOS', 'read_speed_mb_s': None, 'height_mm': None}
  [4/25] Row 5 | ETA: 3.6min | {'bus_type': 'PCIe', 'model_number': 'Asus GeForce GTX 1650 DUAL OC 4GB GDDR5', 'model': 'GeForce GTX 1650 DUAL OC', 'read_speed_mb_s': None, 'height_mm': None}
  [5/25] Row 6 | ETA: 3.0min | {'bus_type': None, 'model_number': None, 'model': 'DataTraveler Vault', 'read_speed_mb_s': None, 'height_mm': None}
  [6/25] Row 8 | ETA: 2.9min | {'bus_type': 'PCIe', 'model_number': 'Gigabyte GeForce GTX 1660 SUPER OC 6GB Dual Fan Graphics Card', 'model': 

## 8. Standard Evaluation

In [40]:
def normalize(val):
    return str(val).lower().strip()

def is_correct_standard(predicted, ground_truth, attribute):
    if predicted is None or str(predicted).strip().lower() in {
            "", "nan", "none", "unknown", "null"}:
        return False
    if attribute in NUMERIC_ATTRIBUTES:
        try:
            p = float(str(predicted).replace(",", "").strip())
            g = float(str(ground_truth).replace(",", "").strip())
            return abs(p - g) / abs(g) <= 0.10 if g != 0 else p == 0
        except:
            pass
    p = normalize(str(predicted))
    g = normalize(str(ground_truth))
    return p == g or p in g or g in p

results = []
for _, task in eval_df.iterrows():
    idx = task["df1_idx"]
    attr = task["attribute"]
    gt = task["ground_truth"]
    if attr in extracted_df.columns and idx in extracted_df.index:
        predicted = extracted_df.loc[idx, attr]
        predicted_str = str(predicted) if pd.notna(predicted) and predicted is not None else "UNKNOWN"
    else:
        predicted_str = "UNKNOWN"
    correct = is_correct_standard(predicted_str, gt, attr)
    unknown = predicted_str.upper() == "UNKNOWN" or predicted_str.lower() in {"nan", "none", "null"}
    results.append({
        "df1_idx": idx,
        "attribute": attr,
        "is_numeric": task["is_numeric"],
        "ground_truth": gt,
        "predicted": predicted_str,
        "correct_standard": correct,
        "unknown": unknown
    })

results_df = pd.DataFrame(results)

print("=" * 55)
print("STANDARD EVALUATION (String/Numeric Match)")
print("=" * 55)
print(f"Overall accuracy: {results_df['correct_standard'].mean():.3f}")
print(f"UNKNOWN rate:     {results_df['unknown'].mean():.3f}")
print(f"Total tasks:      {len(results_df)}")
print()
print(results_df.groupby("attribute").agg(
    total=("correct_standard", "count"),
    correct=("correct_standard", "sum"),
    accuracy=("correct_standard", "mean"),
    unknown_rate=("unknown", "mean")
).round(3).to_string())
print()
print(results_df[["attribute", "ground_truth", "predicted", "correct_standard"]]
      .to_string(index=False))

STANDARD EVALUATION (String/Numeric Match)
Overall accuracy: 0.200
UNKNOWN rate:     0.457
Total tasks:      35

                 total  correct  accuracy  unknown_rate
attribute                                              
bus_type            10        5     0.500         0.400
height_mm            3        0     0.000         1.000
model                1        0     0.000         0.000
model_number        17        2     0.118         0.294
read_speed_mb_s      4        0     0.000         1.000

      attribute                    ground_truth                                                     predicted  correct_standard
   model_number          GV-N3080GAMING OC-10GD                                        GV-N3080GAMING OC-10GD              True
   model_number                CSSD-F960GBMP510                                                       UNKNOWN             False
read_speed_mb_s                          3480.0                                                       UNKNOWN 

## 9. LLM-Based Evaluation (with KB context)

In [41]:
def llm_evaluate(predicted, ground_truth, attribute, cluster_id, kb, eval_model):
    if predicted == "UNKNOWN" or str(predicted).lower() in {"nan", "none", "null"}:
        return "wrong"
    # KB context for this cluster
    kb_matches = kb[kb["cluster_id"] == cluster_id]
    kb_context = ""
    for _, kb_row in kb_matches.head(3).iterrows():
        val = kb_row.get(attribute)
        title = kb_row.get("title", "")
        if pd.notna(val):
            kb_context += f"  - {str(title)[:80]} | {attribute}: {val}\n"
    eval_prompt = f"""You are evaluating a data cleaning prediction for a product database.

Attribute: {attribute}
Ground truth (from knowledge base): {ground_truth}
Predicted value: {predicted}

Knowledge base context (matching products):
{kb_context if kb_context else '  (none)'}

Judge as one of:
- CORRECT: exact match or semantically equivalent (e.g. PCIe Gen3x4 = PCIe 3.0 x4)
- ACCEPTABLE: partially correct or less detailed but not wrong
- WRONG: clearly incorrect

Respond with JUDGMENT:<label> only. Example: JUDGMENT:CORRECT"""
    response = eval_model.invoke([HumanMessage(content=eval_prompt)])
    response_text = response.content.strip().upper()
    if "JUDGMENT:CORRECT" in response_text:
        return "correct"
    elif "JUDGMENT:ACCEPTABLE" in response_text:
        return "acceptable"
    else:
        return "wrong"


print("Running LLM-based evaluation...")
llm_judgments = []
t0 = time.time()
for i, (_, row) in enumerate(results_df.iterrows()):
    cluster_id = query_df.loc[row["df1_idx"], "cluster_id"]
    judgment = llm_evaluate(
        row["predicted"], row["ground_truth"],
        row["attribute"], cluster_id, kb, eval_model
    )
    llm_judgments.append(judgment)
    print(f"  [{i+1}/{len(results_df)}] {row['attribute']:<20} | "
          f"GT: {str(row['ground_truth']):<25} | "
          f"Pred: {str(row['predicted']):<25} | {judgment}")

results_df["llm_judgment"] = llm_judgments
results_df.to_csv("results_exp6_pydi_llm.csv", index=False)
print(f"\nDone in {time.time()-t0:.1f}s — saved to results_exp6_pydi_llm.csv")

Running LLM-based evaluation...
  [1/35] model_number         | GT: GV-N3080GAMING OC-10GD    | Pred: GV-N3080GAMING OC-10GD    | correct
  [2/35] model_number         | GT: CSSD-F960GBMP510          | Pred: UNKNOWN                   | wrong
  [3/35] read_speed_mb_s      | GT: 3480.0                    | Pred: UNKNOWN                   | wrong
  [4/35] read_speed_mb_s      | GT: 226.0                     | Pred: UNKNOWN                   | wrong
  [5/35] bus_type             | GT: PCIe 3.0 x16              | Pred: PCIe                      | acceptable
  [6/35] model_number         | GT: 90YV0CV2-M0NA00           | Pred: Asus GeForce GTX 1650 DUAL OC 4GB GDDR5 | correct
  [7/35] bus_type             | GT: USB 3.0                   | Pred: UNKNOWN                   | wrong
  [8/35] model_number         | GT: GV-N166SOC-6GD            | Pred: Gigabyte GeForce GTX 1660 SUPER OC 6GB Dual Fan Graphics Card | acceptable
  [9/35] bus_type             | GT: PCIe 3.0 x16              | Pred: PC

## 10. Final Comparison

In [42]:
string_acc = results_df["correct_standard"].mean()
llm_correct = (results_df["llm_judgment"] == "correct").mean()
llm_acceptable = (results_df["llm_judgment"].isin(["correct", "acceptable"])).mean()

print("=" * 60)
print("FINAL COMPARISON: Standard vs LLM Evaluation")
print("=" * 60)
print(f"{'Standard (string/numeric match)':<40} {string_acc:.3f}")
print(f"{'LLM eval — correct only':<40} {llm_correct:.3f}")
print(f"{'LLM eval — correct + acceptable':<40} {llm_acceptable:.3f}")

print("\nPer-attribute breakdown:")
per_attr = results_df.groupby("attribute").apply(lambda x: pd.Series({
    "total": len(x),
    "standard_acc": x["correct_standard"].mean(),
    "llm_correct": (x["llm_judgment"] == "correct").mean(),
    "llm_correct+acceptable": (x["llm_judgment"].isin(["correct", "acceptable"])).mean(),
    "unknown_rate": x["unknown"].mean()
})).round(3)
print(per_attr.to_string())

print("\nLLM judgment distribution:")
print(results_df["llm_judgment"].value_counts())

print("\nFull prediction table:")
print(results_df[["attribute", "ground_truth", "predicted",
                   "correct_standard", "llm_judgment"]].to_string(index=False))

FINAL COMPARISON: Standard vs LLM Evaluation
Standard (string/numeric match)          0.200
LLM eval — correct only                  0.086
LLM eval — correct + acceptable          0.457

Per-attribute breakdown:
                 total  standard_acc  llm_correct  llm_correct+acceptable  unknown_rate
attribute                                                                              
bus_type          10.0         0.500        0.000                   0.600         0.400
height_mm          3.0         0.000        0.000                   0.000         1.000
model              1.0         0.000        0.000                   1.000         0.000
model_number      17.0         0.118        0.176                   0.529         0.294
read_speed_mb_s    4.0         0.000        0.000                   0.000         1.000

LLM judgment distribution:
llm_judgment
wrong         19
acceptable    13
correct        3
Name: count, dtype: int64

Full prediction table:
      attribute               

/scratch/ipykernel_3686959/3055839452.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  per_attr = results_df.groupby("attribute").apply(lambda x: pd.Series({
